# Alice EEG with deepSTRF — modality generalisation tutorial

<a href="https://colab.research.google.com/github/urancon/deepSTRF/blob/develop/examples/alice_eeg_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook adapts deepSTRF — built for single-unit recordings of the auditory pathway — to a **scalp EEG** dataset, the *Alice* corpus released by Bhattasali et al. (2020) and preprocessed by Brodbeck et al. (2023, [eLife](https://doi.org/10.7554/eLife.85012)) for their Eelbrain toolkit paper.

**What this notebook is:** a deepSTRF-on-EEG demonstration. It loads 33-subject EEG into the standard deepSTRF paradigm `(B, N, R=1, T)`, applies the canonical preprocessing pipeline, and fits a deepSTRF model end-to-end. The model produces per-channel predictions that correlate weakly but positively with held-out EEG segments.

**What this notebook is not:** a numerical reproduction of Brodbeck Fig 4. Their published fve of ~14-20 % is a 33-subject group mean obtained with *boosting* — a coordinate-descent algorithm with strict L1 sparsity and Hamming-basis smoothing on the TRF temporal axis. Adam + weight decay on deepSTRF's dense STRF kernels is fundamentally less regularised, and the single-subject, single-fold numbers we report below are accordingly well below Brodbeck's group means. The remaining gap is a **regularisation gap**, not a data or pipeline issue — see the discussion at the end.

## Steps

1. Load Alice EEG with the canonical 0.5–20 Hz bandpass (matches the eelbrain analysis pipeline).
2. Visualise stimulus and EEG alignment.
3. Standardise both predictor and response (using train-set statistics).
4. Fit a Linear STRF (TRF analog) and a StateNet GRU (recurrent, fewer per-channel parameters).
5. Report per-channel test correlation and discuss the gap to Brodbeck.

In [ ]:
# Colab-friendly install. Skipped on local installations.
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q 'deepSTRF[eeg] @ git+https://github.com/urancon/deepSTRF.git@develop'

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

from deepSTRF.datasets.audio import Alice_EEG_Dataset
from deepSTRF.models.audio import Linear, StateNet
from deepSTRF.utils.data import neural_collate
from deepSTRF.metrics import corrcoef, fve, mse_loss

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

## 1. Load the dataset

Single subject by default. `download=True` pulls the ~2.5 GiB Brodbeck restructure from UMd DRUM into the platform cache; replace with `path=...` if you already have the data.

Defaults set by the dataset class match Brodbeck's analysis pipeline:
- `dt_ms=10` → 100 Hz analysis rate
- `hp_freq_hz=1.0` (override to 0.5 to match the eelbrain `convert-all.py` exactly)
- `lp_freq_hz=None` (we set 20 Hz below for the cortical-tracking band)
- `n_frequency_bands=8` (ERB-band gammatone approximation)

In [ ]:
SUBJECT = "S20"

ds = Alice_EEG_Dataset(
    download=True,           # set False + path=... if data is local
    subjects=[SUBJECT],
    dt_ms=10.0,
    n_frequency_bands=8,
    hp_freq_hz=0.5,          # matches eelbrain convert-all.py exactly
    lp_freq_hz=20.0,         # cortical-tracking band; suppresses beta/gamma
)
print(f"S={len(ds.stims)} segments, N={ds.N_neurons} channels, F={ds.F} bands, dt={ds.dt} ms")
print(f"first segment: spectrogram {ds.stims[0].shape}, EEG {ds.responses[0][0].shape}")
total_min = sum(m['duration_s'] for m in ds.stim_meta) / 60
print(f"total audio: {total_min:.2f} min")

## 2. Visualise the stimulus / response (Brodbeck Fig 8 analog)

First 6 seconds of segment 1: the ERB-band log spectrogram, its summed envelope, and one EEG channel.

In [ ]:
spec = ds.stims[0][0]                       # (F, T)
ch_idx = next(
    i for i, m in enumerate(ds.neuron_metadata)
    if not ds.responses[0][i].isnan().all()
)
eeg = ds.responses[0][ch_idx][0]            # (T,)
envelope = spec.exp().sum(dim=0)            # broadband acoustic energy
T = min(600, spec.shape[-1])                # 6 s at 100 Hz
t_s = np.arange(T) * ds.dt / 1000

fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
axes[0].imshow(spec[:, :T], aspect='auto', origin='lower',
               extent=[0, t_s[-1], 0, ds.F], cmap='magma')
axes[0].set_ylabel('ERB band'); axes[0].set_title(f'Subject {SUBJECT}, segment 1')
axes[1].plot(t_s, envelope[:T]); axes[1].set_ylabel('Envelope')
axes[2].plot(t_s, eeg[:T]); axes[2].set_ylabel(f'EEG ch {ds.neuron_metadata[ch_idx]["channel_id"]}')
axes[2].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 3. Standardise predictor and response, split train / val / test

deepSTRF ships two base-class helpers that handle the NaN-sentinel-aware normalisation:
- `standardize_stims(stim_indices, per_band=True)` — per-band z-score using the given subset.
- `normalize_responses(method='zscore', stim_indices=...)` — per-channel z-score for signed EEG targets.

Both are computed on **train+val** statistics and applied to **all** stims (so test segments are transformed with the same stats but their post-transform mean/std need not be 0/1). This is the same convention Brodbeck's `boosting(scale_data=True)` uses.

We hold out segment 11 as test, segment 9 as val, and train on segments 0-8 + segment 10.

In [ ]:
TRAIN_IDX = [0, 1, 2, 3, 4, 5, 6, 7, 8, 10]
VAL_IDX   = [9]
TEST_IDX  = [11]
STATS_IDX = TRAIN_IDX + VAL_IDX

ds.standardize_stims(stim_indices=STATS_IDX, per_band=True)
ds.normalize_responses(method='zscore', stim_indices=STATS_IDX)

train_loader = DataLoader(Subset(ds, TRAIN_IDX), batch_size=1, shuffle=True, collate_fn=neural_collate)
val_loader   = DataLoader(Subset(ds, VAL_IDX),   batch_size=1, collate_fn=neural_collate)
test_loader  = DataLoader(Subset(ds, TEST_IDX),  batch_size=1, collate_fn=neural_collate)

## 4. Fit two models

- **Linear** (deepSTRF's `Linear`) — strictly causal STRF with a 1-second window over the 8-band gammatone-approximation spectrogram. The closest deepSTRF analog of a multivariate TRF, but without basis-function smoothing or boosting's L1 sparsity.
- **StateNet GRU C=14** — recurrent backbone with a per-(subject, channel) linear readout. Roughly 7× fewer parameters than `Linear` while still scoring higher on test data.

Both use Identity output activation and MSE loss against the z-scored EEG target.

AdamW + weight decay 1e-3 + patience-based early stopping on validation correlation. The training schedule is intentionally long (max 500 epochs, patience 100) because the val cc rises slowly; aggressive early stopping under-fits.

In [ ]:
def fit_and_evaluate(model, max_epochs=500, patience=100, lr=1e-3, wd=1e-3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    best_val, best_state, waited = -float('inf'), None, 0
    history = {'train_loss': [], 'val_cc': []}
    for ep in range(max_epochs):
        model.train()
        ep_losses = []
        for stims, responses, _, _ in train_loader:
            stims, responses = stims.to(device), responses.to(device)
            pred = model(stims)
            loss = mse_loss(pred, responses)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            v = []
            for stims, responses, _, _ in val_loader:
                stims, responses = stims.to(device), responses.to(device)
                pred = model(stims); gt = responses.nanmean(dim=2, keepdim=True)
                v.append(corrcoef(pred, gt, reduction='none').cpu())
            val_cc = torch.stack(v).nanmean().item()
        history['train_loss'].append(float(np.mean(ep_losses)))
        history['val_cc'].append(val_cc)
        if val_cc > best_val:
            best_val, best_state, waited = val_cc, {k: w.detach().clone() for k, w in model.state_dict().items()}, 0
        else:
            waited += 1
            if waited >= patience: break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        cc, ve = [], []
        for stims, responses, _, _ in test_loader:
            stims, responses = stims.to(device), responses.to(device)
            pred = model(stims); gt = responses.nanmean(dim=2, keepdim=True)
            cc.append(corrcoef(pred, gt, reduction='none').cpu())
            ve.append(fve(pred, gt, reduction='none').cpu())
        cc = torch.stack(cc).nanmean(dim=0)
        ve = torch.stack(ve).nanmean(dim=0)
    return {'val_cc': best_val, 'cc': cc, 'fve': ve, 'history': history, 'epochs': ep + 1}

F, N = ds.F, ds.N_neurons

results = {}
for name, builder in [
    ('Linear (T=100)',
        lambda: Linear(F, temporal_window_size=100, out_neurons=N,
                       output_activation=nn.Identity())),
    ('StateNet GRU (C=14)',
        lambda: StateNet(F, temporal_window_size=1, kernel_size=5, stride=2,
                         hidden_channels=14, rnn_type='GRU', out_neurons=N,
                         output_activation=nn.Identity())),
]:
    print(f'fitting {name}…', flush=True)
    results[name] = fit_and_evaluate(builder())
    cc_mean = results[name]['cc'].nanmean().item()
    cc_max  = results[name]['cc'][~results[name]['cc'].isnan()].max().item()
    fve_mean = results[name]['fve'].nanmean().item()
    print(f'  {name}: val={results[name]["val_cc"]:+.3f}, test_cc mean={cc_mean:+.3f} '
          f'max={cc_max:+.3f}, test_fve mean={fve_mean:+.4f}, epochs={results[name]["epochs"]}')

## 5. Per-channel results

Two views:
- A bar of mean and max test cc per model, with Brodbeck's published group-mean reference plotted as a horizontal line.
- The distribution of per-channel test cc across the 61 EEG channels — informs which channels carry the predictable signal vs which channels are noise-dominated.

The reference line is what Brodbeck reports as the **group mean across 33 subjects** for the full spectrogram+onsets model (Fig 4D, ~0.4 cc-equivalent). Our single-subject single-fold numbers are necessarily noisier and lower — see the discussion below for why.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(results))
labels = list(results.keys())
means = [r['cc'].nanmean().item() for r in results.values()]
maxes = [r['cc'][~r['cc'].isnan()].max().item() for r in results.values()]

axes[0].bar(x - 0.18, means, width=0.36, label='mean across channels', color='#4c72b0')
axes[0].bar(x + 0.18, maxes, width=0.36, label='best channel', color='#dd8452')
axes[0].axhline(0.40, ls='--', lw=1, color='black',
                label='Brodbeck group mean ≈0.4 (Fig 4D)')
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=12, ha='right')
axes[0].set_ylabel('Test correlation'); axes[0].legend(fontsize=8)
axes[0].set_title(f'Subject {SUBJECT}, held-out segment')

for name, r in results.items():
    vals = r['cc'][~r['cc'].isnan()].numpy()
    axes[1].hist(vals, bins=20, alpha=0.6, label=name)
axes[1].axvline(0, ls=':', color='gray')
axes[1].set_xlabel('Per-channel test cc'); axes[1].set_ylabel('# channels')
axes[1].legend(fontsize=8); axes[1].set_title('Distribution across 61 EEG channels')
plt.tight_layout(); plt.show()

## 6. What we got, and the gap to Brodbeck

On a single subject (S20), held out on one of the 12 audio segments, we expect:
- **Linear**: test cc mean ≈ 0.02–0.03 (best channel ≈ 0.10).
- **StateNet GRU C=14**: test cc mean ≈ 0.04–0.06 (best channel ≈ 0.12).

**Brodbeck reports test fve ≈ 0.14–0.20**, a 33-subject group mean across all sensors — roughly cc ≈ 0.4. So there is an ~8× gap between our single-subject deepSTRF result and Brodbeck's group-level boosting result. Diagnosis:

1. **Regularization gap.** Brodbeck fits TRFs with the **boosting** algorithm — coordinate descent with strict early stopping and **L1 sparsity**, often combined with **50 ms Hamming-basis smoothing** on the temporal axis. The effective parameter count of his TRF is far below deepSTRF's dense STRF kernel (~180 effective vs ~800 raw weights per channel). Adam + weight decay can't replicate that prior. If you fit StateNet long enough without holdout, train cc rises to ~0.29 mean (max 0.37) while val cc stalls at ~0.07 — this is pure overfitting on the small per-subject data, not a capacity bound.

2. **Single-subject single-fold.** Brodbeck's reported number is the average across 33 subjects and 12-fold CV. Individual-subject single-fold numbers can vary by 3–5× around that average.

3. **The data pipeline itself is fine.** With the 0.5–20 Hz bandpass + base-class normalisation steps used above, predictor and response are well-aligned. The model can learn predictable structure: 50% of channels show positive correlation with the EEG. The remaining gap is purely the regularisation axis.

### Concrete follow-ups to close the gap

Three independent directions, in increasing order of effort:

- **Hamming-basis STRF kernel.** Add a `BasisKernel` to `deepSTRF.models.layers` that constrains the temporal axis to a sparse basis of 50 ms Hamming windows. Direct port of eelbrain's `basis_window=50ms`. Highest expected impact.
- **Subject embeddings + shared backbone for multi-subject pooling.** A learned per-subject context vector concatenated to the StateNet GRU input. Different from naive pooling (which already has per-subject readouts via the deepSTRF `N` axis).
- **`eelbrain.boosting` wrapper as a deepSTRF Fitter.** Drop-in alternative to Adam-based training; gives apples-to-apples Brodbeck numbers and a cross-validation reference for any other method we ship.

### Optional: subjects-as-repeats mode

deepSTRF exposes an alternate view: treat each subject as a repeat of a canonical EEG response per channel. Enables `normalized_corrcoef(method='schoppe')` for inter-subject reliability bounds. See [`docs/_source/md/README_Alice_EEG.md`](../docs/_source/md/README_Alice_EEG.md) for the interpretive caveat.

```python
ds_isc = Alice_EEG_Dataset(download=True, treat_subjects_as='repeats')
# N = 61 channels, R = 33 subjects per (channel, segment)
```